# Gans — E-Scooter Demand Data Pipeline

**Gans** is a fictional e-scooter sharing startup expanding across Europe. To put scooters
where people actually need them, the operations team needs external context about each city:
how many people live there, what the weather will be, when tourists are flying in, and what
events might draw crowds.

This notebook is my end-to-end **data pipeline** for that problem. It collects data from four
sources, cleans it, and loads it into a relational MySQL database where each piece links back
to a city.

### What it collects

| Source | What it gives us | Why Gans cares |
|---|---|---|
| Wikipedia (web scraping) | City facts: population, area, coordinates | Bigger, denser cities = more demand |
| OpenWeather API | 5-day weather forecast | People skip scooters in the rain |
| AeroDataBox API (via RapidAPI) | Tomorrow's flight arrivals | Tourists land with a backpack and need a ride |
| Ticketmaster API | Upcoming concerts & events | A stadium show is a demand spike |

### The pattern

Every source follows the same four moves, which keeps the code predictable:

> **request** the data -> **parse** the response -> **extract** the useful fields -> **store** it in MySQL

The database is split into linked tables (one `cities` table that everything else references by
`city_id`), so the data stays clean and we can keep a history instead of overwriting it.

> **Note on secrets:** All API keys and the database password live in a `.env` file that is *not*
> committed to the repo. The code reads them by name, so no credentials ever appear in the notebook.


---
## 0 - Setup

All imports and the database connection live here, at the top, so the notebook runs cleanly
from a fresh kernel. If something is "not defined" later, it almost always means this cell
needs to run first.


In [ ]:
import os
import time
from datetime import date, timedelta

import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# Load secrets from the .env file (kept out of version control)
load_dotenv()

WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")
RAPIDAPI_KEY    = os.getenv("RAPIDAPI_KEY")
TICKETMASTER_API_KEY = os.getenv("TICKETMASTER_API_KEY")

# Quick check that the keys loaded (prints True/False, never the keys themselves)
print("Weather key loaded:     ", WEATHER_API_KEY is not None)
print("RapidAPI key loaded:    ", RAPIDAPI_KEY is not None)
print("Ticketmaster key loaded:", TICKETMASTER_API_KEY is not None)

### Database connection

I build a SQLAlchemy `engine` once and reuse it everywhere. The password is URL-encoded with
`quote_plus` so special characters don't break the connection string.


In [ ]:
# Database connection details (the password comes from .env, never hard-coded)
safe_pw = quote_plus(os.getenv("MYSQL_PASSWORD"))
host, port, user, database = "127.0.0.1", 3306, "root", "gans"

engine = create_engine(f"mysql+pymysql://{user}:{safe_pw}@{host}:{port}/{database}")

# Sanity check: ask the DB which tables exist
pd.read_sql("SHOW TABLES", con=engine)

---
# Stage 1 - City facts from Wikipedia

There's no clean API for "give me facts about this city," so I scrape each city's Wikipedia
page. The data I want - population, area, elevation, coordinates - lives in the *infobox*, the
grey table on the right of every city page.

Scraping is brittle, so I built three small, reusable helpers instead of one giant function.
Each does one job, and the bigger functions are composed from them.


### 1.1 - Helper functions

- **`find_label`** locates an infobox row by its label. It accepts a *list* of labels because
  capital cities use *"Sovereign state"* where other cities use *"Country"*.
- **`extract_number`** pulls a clean number out of messy text like `"3,878,100[1]"` - character by
  character, no regex: keep digits and dots, drop commas, allow a leading minus, stop at the first
  letter or bracket.
- **`get_label_value`** finds a row by label and returns the text of its value cell.


In [ ]:
def find_label(soup, labels):
    """Return the first <th> whose text contains any of the given labels."""
    if isinstance(labels, str):
        labels = [labels]
    for th in soup.find_all("th"):
        th_text = th.get_text()
        for label in labels:
            if label in th_text:
                return th
    return None


def extract_number(text):
    """Pull the leading number out of a messy string. Returns a float, or None."""
    if text is None:
        return None
    text = text.strip()
    number = ""
    for char in text:
        if char.isdigit() or char == ".":
            number += char
        elif char == ",":                      # thousands separator - skip it
            continue
        elif char == "-" and number == "":     # leading minus (e.g. below sea level)
            number += char
        else:
            break                              # first "other" character ends the number
    if number in ("", "-", "."):
        return None
    return float(number)


def get_label_value(soup, label):
    """Find an infobox row by label and return the raw text of its value cell."""
    th = find_label(soup, label)
    if th is None:
        return None
    td = th.find_next("td")
    if td is None:
        return None
    return td.get_text()

### 1.2 - Scrape one city

`get_city_data` fetches a single Wikipedia page and pulls out everything I need. Coordinates
come from a hidden `<span class="geo">` that already holds clean decimal values. The country
sometimes carries a footnote marker like `[a]`, so I split it off with `.split("[")[0]`.


In [ ]:
def get_city_data(city):
    """Scrape country, population, area, elevation and coordinates for one city."""
    url = f"https://en.wikipedia.org/wiki/{city}"
    headers = {"User-Agent": "Chrome/134.0.0.0"}   # look like a normal browser
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")

    # Coordinates: the hidden decimal span, e.g. "52.52; 13.405"
    geo_text = soup.find("span", class_="geo").get_text()
    lat, lon = [float(x) for x in geo_text.split(";")]

    # Country: capitals use "Sovereign state", other cities use "Country"
    country_text = get_label_value(soup, ["Country", "Sovereign state"])
    country = None
    if country_text:
        country = country_text.strip().split("[")[0].strip()   # drop footnotes

    # Numeric fields, cleaned by extract_number
    population = extract_number(get_label_value(soup, "Population"))
    area = extract_number(get_label_value(soup, "Area"))
    elevation = extract_number(get_label_value(soup, "Elevation"))
    if population is not None:
        population = int(population)

    return {
        "cityname": city,
        "country": country,
        "population": population,
        "area_km2": area,
        "elevation_m": elevation,
        "latitude": lat,
        "longitude": lon,
    }

### 1.3 - Scrape many cities

`scrape_cities` is the classic *empty-list-then-loop-then-DataFrame* pattern: collect one
dictionary per city, then hand the whole list to pandas.


In [ ]:
def scrape_cities(city_list):
    """Take a list of city names and return a tidy DataFrame of their data."""
    results = []
    for city in city_list:
        results.append(get_city_data(city))
    return pd.DataFrame(results)

### 1.4 - Run the scraper

I picked populous European cities that fit Gans' profile (dense, urban, scooter-friendly), then
added a **density** column - population per km2 - which is a neat demand signal in itself.


In [ ]:
cities = [
    "Berlin", "Hamburg", "Munich", "Cologne", "Frankfurt", "Stuttgart", "Düsseldorf",
    "Paris", "Madrid", "Barcelona", "Rome", "Milan", "Vienna", "Amsterdam",
    "Lisbon", "Brussels", "Prague", "Copenhagen", "Stockholm", "Warsaw",
    "Budapest", "Zurich",
]

cities_df = scrape_cities(cities)
cities_df["density"] = (cities_df["population"] / cities_df["area_km2"]).round(1)
cities_df.head()

---
# Stage 2 - Designing the database & loading cities

### Why split the data into tables?

A single giant table would mix things that change at different rates. Instead:

- **`cities`** holds *static* facts - a city's coordinates and area don't change. One row per city.
- **`populations`** holds *measurements over time*. Population is re-counted periodically, so storing
  it separately with a `recorded_on` date keeps a **history** instead of overwriting.

The two are linked by `city_id`: a **primary key** in `cities`, a **foreign key** in `populations`.
The same idea extends to `weather`, `flights` and `events` later - everything points back to a city.

> The `CREATE TABLE` statements live in a separate `gans_schema.sql` file (run once in a SQL client),
> so this notebook focuses on collecting and loading data.

### The "merge swap"

My DataFrames know each city by **name**, but the tables link by **`city_id`** (which MySQL
auto-generates). So the loading pattern is always:

1. Load `cities` first -> MySQL assigns each a `city_id`.
2. Read those ids back.
3. `merge` them onto the other rows by `cityname` - swapping the name for the id.
4. Insert.


In [ ]:
# --- Load the static city facts ---
cities_table = cities_df[["cityname", "country", "area_km2",
                          "elevation_m", "latitude", "longitude"]]
cities_table.to_sql("cities", con=engine, if_exists="append", index=False)

# --- Load population as a dated measurement ---
saved_cities = pd.read_sql("SELECT city_id, cityname FROM cities", con=engine)
pop_table = cities_df.merge(saved_cities, on="cityname")          # attach city_id
pop_table = pop_table[["city_id", "population"]].copy()
pop_table["recorded_on"] = date.today()                            # tag with today's date
pop_table.to_sql("populations", con=engine, if_exists="append", index=False)

print("Loaded cities and populations.")

**Verify** by joining the two tables back together on `city_id`:

In [ ]:
pd.read_sql("""
    SELECT c.cityname, c.country, p.population, p.recorded_on
    FROM cities c
    JOIN populations p ON c.city_id = p.city_id
    ORDER BY p.population DESC
""", con=engine).head()

---
# Stage 3 - Weather forecasts (OpenWeather API)

Rain kills scooter demand, so a 5-day forecast for each city is genuinely useful. OpenWeather's
free *5-day / 3-hour* endpoint returns ~40 forecast points per city (every 3 hours for 5 days).

This is my first real API. Unlike scraping, the answer is clean JSON. Two things to note:

- The key goes in the **URL params** (`appid`), not in headers.
- Weather is a **time series** - ~40 rows per city is correct, *not* duplication. Each row is the
  same city at a different timestamp.


In [ ]:
def get_weather(lat, lon, api_key):
    """Fetch the 5-day / 3-hour forecast for one location. Returns a DataFrame."""
    url = "https://api.openweathermap.org/data/2.5/forecast"
    params = {"lat": lat, "lon": lon, "appid": api_key, "units": "metric"}
    data = requests.get(url, params=params).json()

    forecasts = []
    for forecast in data["list"]:
        forecasts.append({
            "forecast_time":    forecast.get("dt_txt"),
            "temperature":      forecast.get("main", {}).get("temp"),
            "feels_like":       forecast.get("main", {}).get("feels_like"),
            "humidity":         forecast.get("main", {}).get("humidity"),
            "description":      forecast.get("weather", [{}])[0].get("description"),
            "wind_speed":       forecast.get("wind", {}).get("speed"),
            "rain_probability": forecast.get("pop", 0),
            "rain_mm":          forecast.get("rain", {}).get("3h", 0),  # absent on dry days
        })
    return pd.DataFrame(forecasts)

### A reusable refresh function

Rather than scatter the collection across cells, I wrapped the whole thing - *clear old data,
read the cities, loop, store* - into one `update_weather()` call. Running it always leaves the
`weather` table holding the **latest** forecast (it wipes the old rows first, so re-running never
piles up duplicates). This is also exactly the shape a scheduled cloud job would need.


In [ ]:
def update_weather(engine, api_key):
    """Refresh the weather table: wipe old data, collect fresh, store. Safe to re-run."""

    # 1. Clear old forecasts so we never accumulate duplicates
    with engine.connect() as conn:
        conn.execute(text("DELETE FROM weather"))
        conn.commit()

    # 2. Read the cities (with ids + coordinates) straight from the database
    cities = pd.read_sql(
        "SELECT city_id, cityname, latitude, longitude FROM cities", con=engine
    )

    # 3. Collect weather for each city, tagging rows with city_id directly
    all_weather = []
    for _, row in cities.iterrows():
        city_weather = get_weather(row["latitude"], row["longitude"], api_key)
        city_weather["city_id"] = row["city_id"]
        all_weather.append(city_weather)

    weather_df = pd.concat(all_weather, ignore_index=True)

    # 4. Keep the table's columns and store
    weather_df = weather_df[["city_id", "forecast_time", "temperature", "feels_like",
                             "humidity", "description", "wind_speed",
                             "rain_probability", "rain_mm"]]
    weather_df.to_sql("weather", con=engine, if_exists="append", index=False)
    return weather_df

In [ ]:
fresh_weather = update_weather(engine, WEATHER_API_KEY)
print("Stored", len(fresh_weather), "weather rows.")

In [ ]:
pd.read_sql("""
    SELECT c.cityname, w.forecast_time, w.temperature, w.description
    FROM cities c
    JOIN weather w ON c.city_id = w.city_id
    LIMIT 10
""", con=engine)

---
# Stage 4 - Flight arrivals (AeroDataBox via RapidAPI)

Tourists fly in with a backpack and need a ride from the airport - a textbook scooter customer.
So I collect tomorrow's arrivals for each city's main airport using AeroDataBox, accessed through
the **RapidAPI** marketplace.

Three things that made this the trickiest source:

1. **The key goes in headers**, not params - RapidAPI's convention (`x-rapidapi-key`, `x-rapidapi-host`).
2. **Tomorrow's date is computed**, never hard-coded, so the pipeline keeps working when it runs daily.
   The endpoint wants a `YYYY-MM-DDTHH:MM` time window.
3. **The arrival time is nested** as `movement -> scheduledTime -> local` - not a flat field. Finding
   that took inspecting one raw response.

> **Free-tier note:** the free plan is rate-limited, so the collection loop pauses one second between
> calls (`time.sleep`). Without the pause, rapid back-to-back calls silently return empty. I also keep
> the city list small to respect the monthly request budget.


In [ ]:
def get_arrivals(iata, api_key):
    """Get tomorrow morning's arrivals at one airport (by IATA code). Returns a DataFrame."""
    tomorrow = date.today() + timedelta(days=1)
    url = (
        f"https://aerodatabox.p.rapidapi.com/flights/airports/iata/"
        f"{iata}/{tomorrow}T00:00/{tomorrow}T12:00"
    )
    params = {"direction": "Arrival", "withCancelled": "false"}
    headers = {
        "x-rapidapi-key": api_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
    }

    data = requests.get(url, headers=headers, params=params).json()

    arrivals = []
    for flight in data.get("arrivals", []):
        movement = flight.get("movement", {})
        origin   = movement.get("airport", {})
        sched    = movement.get("scheduledTime", {})   # nested sub-dict with local/utc

        arrivals.append({
            "arrival_airport_iata": iata,
            "arrival_time":         sched.get("local"),
            "flight_number":        flight.get("number"),
            "airline":              flight.get("airline", {}).get("name"),
            "origin_airport":       origin.get("name"),
            "origin_iata":          origin.get("iata"),
            "aircraft":             flight.get("aircraft", {}).get("model"),
        })
    return pd.DataFrame(arrivals)

### Map cities to airports, then loop

I demonstrate the pipeline on a focused set of high-traffic, tourist-heavy airports (keeping the
request count well within the free tier).


In [ ]:
city_airports = {
    "Berlin":    "BER",
    "Munich":    "MUC",
    "Frankfurt": "FRA",
    "Paris":     "CDG",
    "Barcelona": "BCN",
}

all_arrivals = []
for city, iata in city_airports.items():
    city_flights = get_arrivals(iata, RAPIDAPI_KEY)
    city_flights["cityname"] = city
    print(f"{city} ({iata}): {len(city_flights)} flights")
    all_arrivals.append(city_flights)
    time.sleep(1)   # respect the free-tier rate limit

flights_df = pd.concat(all_arrivals, ignore_index=True)
print("Total:", flights_df.shape)

### Clean and load

The API returns times like `2026-05-29 06:25+02:00`. MySQL's `DATETIME` doesn't accept the
`+02:00` timezone suffix, so I keep the first 16 characters (`YYYY-MM-DD HH:MM`) before loading.


In [ ]:
# Strip the timezone offset so MySQL DATETIME accepts it
flights_df["arrival_time"] = flights_df["arrival_time"].str[:16]

# Merge-swap cityname -> city_id, then load
saved_cities = pd.read_sql("SELECT city_id, cityname FROM cities", con=engine)
flights_to_load = flights_df.merge(saved_cities, on="cityname")
flights_to_load = flights_to_load[["city_id", "arrival_time", "flight_number",
                                   "airline", "origin_airport", "origin_iata", "aircraft"]]
flights_to_load.to_sql("flights", con=engine, if_exists="append", index=False)
print("Loaded", len(flights_to_load), "flights.")

In [ ]:
pd.read_sql("""
    SELECT c.cityname, f.arrival_time, f.airline, f.origin_airport
    FROM cities c
    JOIN flights f ON c.city_id = f.city_id
    LIMIT 10
""", con=engine)

---
# Stage 5 - Events (Ticketmaster Discovery API)

A stadium concert or festival can send scooter demand soaring around the venue. Ticketmaster's
Discovery API lists upcoming events by city - and it's the friendliest API here: key in the URL
params (like OpenWeather), and a generous free tier.

The useful fields are nested in a few places: the date sits in `dates.start`, the category in the
first `classifications` entry, and the venue inside `_embedded.venues`.


In [ ]:
def get_events(city, api_key):
    """Get upcoming events for one city. Returns a DataFrame."""
    url = "https://app.ticketmaster.com/discovery/v2/events.json"
    params = {"apikey": api_key, "city": city, "size": 50, "sort": "date,asc"}

    data = requests.get(url, params=params).json()

    events = []
    for event in data.get("_embedded", {}).get("events", []):
        dates  = event.get("dates", {}).get("start", {})
        cls    = event.get("classifications", [{}])[0]      # list -> take first
        venue  = event.get("_embedded", {}).get("venues", [{}])[0]

        events.append({
            "event_name": event.get("name"),
            "event_date": dates.get("localDate"),
            "event_time": dates.get("localTime"),
            "category":   cls.get("segment", {}).get("name"),
            "genre":      cls.get("genre", {}).get("name"),
            "venue_name": venue.get("name"),
            "venue_city": venue.get("city", {}).get("name"),
        })
    return pd.DataFrame(events)

In [ ]:
all_events = []
for city in city_airports:           # looping a dict yields its keys (the city names)
    city_events = get_events(city, TICKETMASTER_API_KEY)
    city_events["cityname"] = city
    print(f"{city}: {len(city_events)} events")
    all_events.append(city_events)
    time.sleep(1)

events_df = pd.concat(all_events, ignore_index=True)
print("Total:", events_df.shape)

### Clean and load

Some events are missing a time or venue. pandas stores those gaps as `NaN`, which MySQL doesn't
like, so I convert them to `None` (SQL `NULL`) before loading.


In [ ]:
# Turn pandas NaN into None so MySQL accepts the gaps as NULL
events_df = events_df.where(pd.notnull(events_df), None)

saved_cities = pd.read_sql("SELECT city_id, cityname FROM cities", con=engine)
events_to_load = events_df.merge(saved_cities, on="cityname")
events_to_load = events_to_load[["city_id", "event_name", "event_date", "event_time",
                                 "category", "genre", "venue_name", "venue_city"]]
events_to_load.to_sql("events", con=engine, if_exists="append", index=False)
print("Loaded", len(events_to_load), "events.")

In [ ]:
pd.read_sql("""
    SELECT c.cityname, e.event_name, e.event_date, e.category
    FROM cities c
    JOIN events e ON c.city_id = e.city_id
    LIMIT 10
""", con=engine)

---
# Wrap-up & next steps

At this point the database holds **five linked tables**, all keyed back to a city:

```
cities --+-- populations   (population over time)
         +-- weather       (5-day forecast per city)
         +-- flights       (tomorrow's arrivals)
         +-- events        (upcoming concerts & shows)
```

### What I'd improve next

- **Events coverage by coordinates.** Ticketmaster's `city` filter misses venues filed under
  suburb names (Frankfurt/Paris came back light). Searching by the lat/long I already have, with a
  radius, would catch more.
- **More airports per city.** Paris also has Orly (ORY); adding second airports gives fuller arrival counts.

### Stage 6 - automation (the real goal)

The functions are written to be re-runnable on their own, which is exactly what a scheduled job
needs. The production version would:

1. Move the database to the cloud .
2. Package the collectors (`update_weather`, the flights and events loops) into a single handler.
3. Deploy it as a **cloud function** (AWS Lambda / Google Cloud Functions).
4. Trigger it on a daily **schedule**, so fresh data flows in with no manual steps.
